In [1]:
import math
import platform
import warnings

import torch as pt
import torch
import torch.nn as nn
from PIL import Image
from typing import Optional, Tuple, Dict,List
from omegaconf import DictConfig,OmegaConf,open_dict

from uuid import uuid1
import os

In [ ]:
import sys
sys.path.append("../../")
from lynadapter.warp_load_save import load,save,load_kernel,save_kernel

spike_func = lambda _: torch.gt(_, 0.).to(_.dtype)

MEMB_MODE = (0, pt.relu)
SOMA_PARAMS = {'alpha': .3, 'beta': 0., 'theta': .5, 'v_0': 0., 'shape': [], 'learn': False}


class Lif(nn.Module):
    def __init__(self,
                 norm: callable,  # state normalizer
                 mode: str,  # spike | analog
                 memb_mode: tuple,  # (0~3, pt.relu|sigmoid|tanh), valid when ``mode==analog``
                 soma_params: dict,  # {alpha, beta, theta, v_0, shape, learn}
                 noise: float,  # noise scale in training
                 on_apu: bool  # for lyngor apu
                 ):
        super(Lif, self).__init__()
        assert mode in ['spike', 'analog']
        assert len(memb_mode) == 2 and memb_mode[0] in range(4)

        self.norm = norm
        self.mode = mode
        self.memb_mode = memb_mode

        shape = soma_params['shape']
        assert isinstance(shape, (list, tuple))
        learn = soma_params['learn']
        func = lambda _: nn.Parameter(pt.randn(shape) if _ is None else (pt.ones(shape) * _), requires_grad=learn)

        self.alpha = func(soma_params['alpha'])
        self.beta = func(soma_params['beta'])
        self.theta = func(soma_params['theta'])
        self.v_0 = func(soma_params['v_0'])

        self.noise = noise
        if on_apu:
            self.id = uuid1()

        self.v = pt.tensor(0.)#None

    def forward(self, x: pt.Tensor, h_and_c_previous: Optional[Tuple[pt.Tensor, pt.Tensor]] = None) -> pt.Tensor:
        v = None
        if hasattr(self, 'id'):  
            v = load(self.v.clone(), f'v{self.id}',mode=0,init_zero_use_data=x)
        else:
            v = self.v  # .clone()

        # integration
        v_upd = x + v
        # homeostasis
        if self.norm:
            v_upd = self.norm(v_upd)

        v = v_upd

        if self.training and self.noise > 0:
            self.add_noise()

        v_ = v - self.theta

        # spike firing
        fire = spike_func(v_)
        fire_inv = 1. - fire
        # reset
        v = fire * self.v_0 + fire_inv * v
        # leakage
        v = self.alpha * v + self.beta

        oupt = pt.tanh(v)

        if hasattr(self, 'id'):
            # self.v = v.clone()
            self.v = v
            save(self.v, f'v{self.id}')
        else:
            self.v = v  # .clone()

        return oupt

    def add_noise(self):
        with pt.no_grad():
            v_shape = self.v.shape
            noise = pt.randn(v_shape, dtype=self.v.dtype, device=self.v.device)
            scale = pt.std(self.v, dim=[_ for _ in range(2, len(v_shape))], keepdim=True) * self.noise
            self.v += noise * scale

    def reset(self, xi):
        self.v = pt.zeros_like(xi)

In [4]:
class Lif2d(Lif):

    def __init__(self,
                 norm_state=False,
                 mode='spike', memb_mode=MEMB_MODE, soma_params=SOMA_PARAMS, noise=0, hidden_channel=None, on_apu=True
                 ):
        norm = nn.BatchNorm2d(hidden_channel) if norm_state else None
        global SOMA_PARAMS
        if soma_params == 'all_share':
            SOMA_PARAMS = SOMA_PARAMS
        elif soma_params == 'channel_share':
            SOMA_PARAMS['shape'] = [1, hidden_channel, 1, 1]
        else:
            raise NotImplemented
        super(Lif2d, self).__init__(norm, mode, memb_mode, SOMA_PARAMS, noise, on_apu)

In [5]:
def autopad(k, p=None, d=1):  # kernel, padding, dilation
    # Pad to 'same' shape outputs
    if d > 1:
        k = d * (k - 1) + 1 if isinstance(k, int) else [d * (x - 1) + 1 for x in k]  # actual kernel-size
    if p is None:
        p = k // 2 if isinstance(k, int) else [x // 2 for x in k]  # auto-pad
    return p

In [6]:
class Conv(nn.Module):
    # Standard convolution with args(ch_in, ch_out, kernel, stride, padding, groups, dilation, activation)
    default_act = nn.SiLU()  # default activation
    # default_act = nn.ReLU()

    def __init__(self, c1, c2, k=1, s=1, p=None, g=1, d=1, act=True):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, autopad(k, p, d), groups=g, dilation=d, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = self.default_act if act is True else act if isinstance(act, nn.Module) else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

    def forward_fuse(self, x):
        return self.act(self.conv(x))

In [7]:
class Conv_lif(nn.Module):
    # Standard convolution with args(ch_in, ch_out, kernel, stride, padding, groups, dilation, activation)
    # default_act = nn.SiLU()  # default activation

    def __init__(self, c1, c2, k=1, s=1, p=None, g=1, d=1, act=True):
        super().__init__()
        self.default_act = Lif2d(norm_state=True, mode='analog', memb_mode=(0, torch.tanh),
                                 soma_params='channel_share', hidden_channel=c2)
        self.conv = nn.Conv2d(c1, c2, k, s, autopad(k, p, d), groups=g, dilation=d, bias=False)
        # self.bn = nn.BatchNorm2d(c2)
        self.act = self.default_act if act is True else act if isinstance(act, nn.Module) else nn.Identity()

    def forward(self, x):
        # output, states = self.act(self.conv(x))
        output = self.act(self.conv(x))
        return output

    def forward_fuse(self, x):
        return self.act(self.conv(x))

In [8]:
class Bottleneck_common(nn.Module):
    # Standard bottleneck
    def __init__(self, c1, c2, shortcut=True, g=1, e=0.5):  # ch_in, ch_out, shortcut, groups, expansion
        super().__init__()
        c_ = int(c2 * e)  # hidden channels
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(c_, c2, 3, 1, g=g)
        self.add = shortcut and c1 == c2

    def forward(self, x):
        return x + self.cv2(self.cv1(x)) if self.add else self.cv2(self.cv1(x))

In [9]:
class Bottleneck(nn.Module):
    # Standard bottleneck
    def __init__(
        self,
        in_channels,
        out_channels,
        shortcut=True,
        expansion=0.5,
        depthwise=False,
        act="silu",
    ):
        super().__init__()
        hidden_channels = int(out_channels * expansion)
        Conv = DWConv if depthwise else BaseConv
        self.conv1 = BaseConv(in_channels, hidden_channels, 1, stride=1, act=act)
        self.conv2 = Conv(hidden_channels, out_channels, 3, stride=1, act=act)
        self.use_add = shortcut and in_channels == out_channels

    def forward(self, x):
        y = self.conv2(self.conv1(x))
        if self.use_add:
            y = y + x
        return y

In [10]:
class C3(nn.Module):
    # CSP Bottleneck with 3 convolutions
    def __init__(self, c1, c2, n=1, shortcut=True, g=1, e=0.5, lif=False):  # ch_in, ch_out, number, shortcut, groups, expansion
        super().__init__()
        self.lif = lif
        c_ = int(c2 * e)  # hidden channels
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(c1, c_, 1, 1)
        if self.lif:
            self.cv3 = Conv_lif(2 * c_, c2, 1)
        else:
            self.cv3 = Conv(2 * c_, c2, 1)  # optional act=FReLU(c2)
        self.m = nn.Sequential(*(Bottleneck_common(c_, c_, shortcut, g, e=1.0) for _ in range(n)))

    def forward(self, x):
        # if self.lif:
        #     output, states = self.cv3(torch.cat((self.m(self.cv1(x)), self.cv2(x)), 1))
        #     return output, states
        # else:
        return self.cv3(torch.cat((self.m(self.cv1(x)), self.cv2(x)), 1))

In [11]:
class SPPF(nn.Module):
    # Spatial Pyramid Pooling - Fast (SPPF) layer for YOLOv5 by Glenn Jocher
    def __init__(self, c1, c2, k=5, lif=False):  # equivalent to SPP(k=(5, 9, 13))
        super().__init__()
        self.lif = lif
        c_ = c1 // 2  # hidden channels
        self.cv1 = Conv(c1, c_, 1, 1)
        if self.lif:
            self.cv2 = Conv_lif(c_ * 4, c2, 1, 1)
        else:
            self.cv2 = Conv(c_ * 4, c2, 1, 1)
        self.m = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)

    def forward(self, x):
        x = self.cv1(x)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')  # suppress torch 1.9.0 max_pool2d() warning
            y1 = self.m(x)
            y2 = self.m(y1)
            # if self.lif:
            #     output, states = self.cv2(torch.cat((x, y1, y2, self.m(y2)), 1))
            #     return output, states
            # else:
            return self.cv2(torch.cat((x, y1, y2, self.m(y2)), 1))

In [ ]:
class YoloDetS(nn.Module):
    """
    Spiking YOLO Detection Network (S-YOLO).
    
    A spiking neural network variant of YOLO for object detection,
    incorporating LIF (Leaky Integrate-and-Fire) neurons for efficient
    event-based processing.
    """
    def __init__(self, in_channels=20):
        super().__init__()
        self.num_stages = 4
        self.conv1 = Conv(in_channels, 32, 6, 2, 1)
        self.conv2 = Conv_lif(32, 64, 3, 2, 1)

        self.C3_1 = C3(64, 64, 1)
        self.conv3 = Conv_lif(64, 128, 3, 2, 1)

        self.C3_2 = C3(128, 128, 2)
        self.conv4 = Conv(128, 256, 3, 2, 1)
        self.C3_3 = C3(256, 256, 3, lif=True)

        self.conv5 = Conv(256, 512, 3, 2, 1)
        self.C3_4 = C3(512, 512, 1)
        self.SPPF = SPPF(512, 512, lif=True)
        self._register_lyn_reset_hook()

    def _register_lyn_reset_hook(self):
        """
        Register forward pre-hooks for all LIF neuron modules.
        This enables automatic state reset during inference on APU.
        """
        for child in self.modules():
            if isinstance(child, Lif2d):  # Lif, Lif1d, Conv2dLif, FcLif...
                assert hasattr(child, 'reset')
                child.register_forward_pre_hook(self.lyn_reset_hook)

    @staticmethod
    def lyn_reset_hook(m, xi: tuple):
        """
        Hook function to reset LIF neuron state at the first timestep.
        
        Args:
            m: The LIF module
            xi: Input tensor (wrapped in tuple)
        """
        assert isinstance(xi, tuple) and len(xi) == 1
        xi = xi[0]
        if not hasattr(m, 'lyn_cnt'):
            setattr(m, 'lyn_cnt', 0)
        if m.lyn_cnt == 0:
            # print(m)
            m.reset(xi)
            m.lyn_cnt += 1
        else:
            m.lyn_cnt += 1

    def forward(self, x: pt.Tensor):

        output: BackboneFeatures = list()

        x = self.conv1(x)
        x = self.conv2(x)
        output.append(x)

        x = self.C3_1(x)
        x = self.conv3(x)
        output.append(x)

        x = self.C3_2(x)
        x = self.conv4(x)
        x = self.C3_3(x)
        output.append(x)

        x = self.conv5(x)
        x = self.C3_4(x)
        x = self.SPPF(x)
        output.append(x)

        return output[0], output[1], output[2], output[3]

    def _reset_lyn_cnt(self):
        """Reset the timestep counter for all LIF modules."""
        for child in self.modules():
            if hasattr(child, 'lyn_cnt'):
                child.lyn_cnt = 0

In [13]:
def get_activation(name="silu", inplace=True):
    if name == "silu":
        module = nn.SiLU(inplace=inplace)
    elif name == "relu":
        module = nn.ReLU(inplace=inplace)
    elif name == "lrelu":
        module = nn.LeakyReLU(0.1, inplace=inplace)
    else:
        raise AttributeError("Unsupported act type: {}".format(name))
    return module

In [14]:
class CSPLayer(nn.Module):
    """C3 in yolov5, CSP Bottleneck with 3 convolutions"""

    def __init__(
        self,
        in_channels,
        out_channels,
        n=1,
        shortcut=True,
        expansion=0.5,
        depthwise=False,
        act="silu",
    ):
        """
        Args:
            in_channels (int): input channels.
            out_channels (int): output channels.
            n (int): number of Bottlenecks. Default value: 1.
        """
        # ch_in, ch_out, number, shortcut, groups, expansion
        super().__init__()
        hidden_channels = int(out_channels * expansion)  # hidden channels
        self.conv1 = BaseConv(in_channels, hidden_channels, 1, stride=1, act=act)
        self.conv2 = BaseConv(in_channels, hidden_channels, 1, stride=1, act=act)
        self.conv3 = BaseConv(2 * hidden_channels, out_channels, 1, stride=1, act=act)
        module_list = [
            Bottleneck(
                hidden_channels, hidden_channels, shortcut, 1.0, depthwise, act=act
            )
            for _ in range(n)
        ]
        self.m = nn.Sequential(*module_list)

    def forward(self, x):
        x_1 = self.conv1(x)
        x_2 = self.conv2(x)
        x_1 = self.m(x_1)
        x = torch.cat((x_1, x_2), dim=1)
        return self.conv3(x)

In [15]:
class BaseConv(nn.Module):
    """A Conv2d -> Batchnorm -> silu/leaky relu block"""

    def __init__(
        self, in_channels, out_channels, ksize, stride, groups=1, bias=False, act="silu"
    ):
        super().__init__()
        # same padding
        pad = (ksize - 1) // 2
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=ksize,
            stride=stride,
            padding=pad,
            groups=groups,
            bias=bias,
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = get_activation(act, inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

    def fuseforward(self, x):
        return self.act(self.conv(x))

In [16]:
class YOLOPAFPN_apu(nn.Module):
    """
    Removed the direct dependency on the backbone.
    """

    def __init__(
            self,
            depth: float = 1.0,
            in_stages: Tuple[int, ...] = (0, 1, 2),
            in_channels: Tuple[int, ...] = (256, 512, 1024),
            depthwise: bool = False,
            act: str = "silu",
            compile_cfg: Optional[Dict] = None,
    ):
        super().__init__()
        assert len(in_stages) == len(in_channels)
        assert len(in_channels) == 3, 'Current implementation only for 3 feature maps'
        self.in_features = in_stages
        self.in_channels = in_channels
        Conv = DWConv if depthwise else BaseConv

        ###### Compile if requested ######
        if compile_cfg is not None:
            compile_mdl = compile_cfg['enable']
            if compile_mdl and th_compile is not None:
                self.forward = th_compile(self.forward, **compile_cfg['args'])
            elif compile_mdl:
                print('Could not compile PAFPN because torch.compile is not available')

        ##################################

        # self.upsample = lambda x: nn.functional.interpolate(x, scale_factor=2, mode='nearest-exact')
        self.upsample = pt.nn.UpsamplingNearest2d(scale_factor=2)
        self.lateral_conv0 = BaseConv(
            in_channels[2], in_channels[1], 1, 1, act=act
        )
        self.C3_p4 = CSPLayer(
            2 * in_channels[1],
            in_channels[1],
            round(3 * depth),
            False,
            depthwise=depthwise,
            act=act,
        )  # cat

        self.reduce_conv1 = BaseConv(
            in_channels[1], in_channels[0], 1, 1, act=act
        )
        self.C3_p3 = CSPLayer(
            2 * in_channels[0],
            in_channels[0],
            round(3 * depth),
            False,
            depthwise=depthwise,
            act=act,
        )

        # bottom-up conv
        self.bu_conv2 = Conv(
            in_channels[0], in_channels[0], 3, 2, act=act
        )
        self.C3_n3 = CSPLayer(
            2 * in_channels[0],
            in_channels[1],
            round(3 * depth),
            False,
            depthwise=depthwise,
            act=act,
        )

        # bottom-up conv
        self.bu_conv1 = Conv(
            in_channels[1], in_channels[1], 3, 2, act=act
        )
        self.C3_n4 = CSPLayer(
            2 * in_channels[1],
            in_channels[2],
            round(3 * depth),
            False,
            depthwise=depthwise,
            act=act,
        )

        ###### Compile if requested ######
        if compile_cfg is not None:
            compile_mdl = compile_cfg['enable']
            if compile_mdl and th_compile is not None:
                self.forward = th_compile(self.forward, **compile_cfg['args'])
            elif compile_mdl:
                print('Could not compile PAFPN because torch.compile is not available')
        ##################################

    def forward(self, input):
        """
        Args:
            inputs: Feature maps from backbone

        Returns:
            Tuple[Tensor]: FPN feature.
        """
        features = [input[f] for f in self.in_features]
        x2, x1, x0 = features

        fpn_out0 = self.lateral_conv0(x0)  # 1024->512/32
        f_out0 = self.upsample(fpn_out0)  # 512/16
        f_out0 = pt.cat([f_out0, x1], 1)  # 512->1024/16
        f_out0 = self.C3_p4(f_out0)  # 1024->512/16

        fpn_out1 = self.reduce_conv1(f_out0)  # 512->256/16
        f_out1 = self.upsample(fpn_out1)  # 256/8
        f_out1 = pt.cat([f_out1, x2], 1)  # 256->512/8
        pan_out2 = self.C3_p3(f_out1)  # 512->256/8

        p_out1 = self.bu_conv2(pan_out2)  # 256->256/16
        p_out1 = pt.cat([p_out1, fpn_out1], 1)  # 256->512/16
        pan_out1 = self.C3_n3(p_out1)  # 512->512/16

        p_out0 = self.bu_conv1(pan_out1)  # 512->512/32
        p_out0 = pt.cat([p_out0, fpn_out0], 1)  # 512->1024/32
        pan_out0 = self.C3_n4(p_out0)  # 1024->1024/32

        outputs = [pan_out2, pan_out1, pan_out0]
        return outputs

In [17]:
def bboxes_iou(bboxes_a, bboxes_b, xyxy=True):
    if bboxes_a.shape[1] != 4 or bboxes_b.shape[1] != 4:
        raise IndexError

    if xyxy:
        tl = torch.max(bboxes_a[:, None, :2], bboxes_b[:, :2])
        br = torch.min(bboxes_a[:, None, 2:], bboxes_b[:, 2:])
        area_a = torch.prod(bboxes_a[:, 2:] - bboxes_a[:, :2], 1)
        area_b = torch.prod(bboxes_b[:, 2:] - bboxes_b[:, :2], 1)
    else:
        tl = torch.max(
            (bboxes_a[:, None, :2] - bboxes_a[:, None, 2:] / 2),
            (bboxes_b[:, :2] - bboxes_b[:, 2:] / 2),
        )
        br = torch.min(
            (bboxes_a[:, None, :2] + bboxes_a[:, None, 2:] / 2),
            (bboxes_b[:, :2] + bboxes_b[:, 2:] / 2),
        )

        area_a = torch.prod(bboxes_a[:, 2:], 1)
        area_b = torch.prod(bboxes_b[:, 2:], 1)
    en = (tl < br).type(tl.type()).prod(dim=2)
    area_i = torch.prod(br - tl, 2) * en  # * ((tl < br).all())
    return area_i / (area_a[:, None] + area_b - area_i)

In [18]:
class BaseConv(nn.Module):
    """A Conv2d -> Batchnorm -> silu/leaky relu block"""

    def __init__(
        self, in_channels, out_channels, ksize, stride, groups=1, bias=False, act="silu"
    ):
        super().__init__()
        # same padding
        pad = (ksize - 1) // 2
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=ksize,
            stride=stride,
            padding=pad,
            groups=groups,
            bias=bias,
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = get_activation(act, inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

    def fuseforward(self, x):
        return self.act(self.conv(x))


class DWConv(nn.Module):
    """Depthwise Conv + Conv"""

    def __init__(self, in_channels, out_channels, ksize, stride=1, act="silu"):
        super().__init__()
        self.dconv = BaseConv(
            in_channels,
            in_channels,
            ksize=ksize,
            stride=stride,
            groups=in_channels,
            act=act,
        )
        self.pconv = BaseConv(
            in_channels, out_channels, ksize=1, stride=1, groups=1, act=act
        )

    def forward(self, x):
        x = self.dconv(x)
        return self.pconv(x)

In [19]:
class YOLOXHead_apu(nn.Module):
    def __init__(
            self,
            num_classes=2,   # for build ,num_classes=80
            strides=(8, 16, 32),
            in_channels=(128, 256, 512),   # 256, 512, 1024
            act="silu",
            depthwise=False,
            compile_cfg: Optional[Dict] = None,
    ):
        super().__init__()

        self.num_classes = num_classes
        self.decode_in_inference = True  # for deploy, set to False

        self.cls_convs = nn.ModuleList()
        self.reg_convs = nn.ModuleList()
        self.cls_preds = nn.ModuleList()
        self.reg_preds = nn.ModuleList()
        self.obj_preds = nn.ModuleList()
        self.stems = nn.ModuleList()
        Conv = DWConv if depthwise else BaseConv

        self.output_strides = None
        self.output_grids = None

        # Automatic width scaling according to original YoloX channel dims.
        # in[-1]/out = 4/1
        # out = in[-1]/4 = 256 * width
        # -> width = in[-1]/1024
        largest_base_dim_yolox = 1024
        largest_base_dim_from_input = in_channels[-1]
        width = largest_base_dim_from_input/largest_base_dim_yolox

        hidden_dim = int(256*width)

        for i in range(len(in_channels)):
            self.stems.append(
                BaseConv(
                    in_channels=in_channels[i],
                    out_channels=hidden_dim,
                    ksize=1,
                    stride=1,
                    act=act,
                )
            )
            self.cls_convs.append(
                nn.Sequential(
                    *[
                        Conv(
                            in_channels=hidden_dim,
                            out_channels=hidden_dim,
                            ksize=3,
                            stride=1,
                            act=act,
                        ),
                        Conv(
                            in_channels=hidden_dim,
                            out_channels=hidden_dim,
                            ksize=3,
                            stride=1,
                            act=act,
                        ),
                    ]
                )
            )
            self.reg_convs.append(
                nn.Sequential(
                    *[
                        Conv(
                            in_channels=hidden_dim,
                            out_channels=hidden_dim,
                            ksize=3,
                            stride=1,
                            act=act,
                        ),
                        Conv(
                            in_channels=hidden_dim,
                            out_channels=hidden_dim,
                            ksize=3,
                            stride=1,
                            act=act,
                        ),
                    ]
                )
            )
            self.cls_preds.append(
                nn.Conv2d(
                    in_channels=hidden_dim,
                    out_channels=self.num_classes,
                    kernel_size=1,
                    stride=1,
                    padding=0,
                )
            )
            self.reg_preds.append(
                nn.Conv2d(
                    in_channels=hidden_dim,
                    out_channels=4,
                    kernel_size=1,
                    stride=1,
                    padding=0,
                )
            )
            self.obj_preds.append(
                nn.Conv2d(
                    in_channels=hidden_dim,
                    out_channels=1,
                    kernel_size=1,
                    stride=1,
                    padding=0,
                )
            )

        self.use_l1 = False    
        
        self.strides = strides
        self.grids = [torch.zeros(1)] * len(in_channels)

        # According to Focal Loss paper:
        self.initialize_biases(prior_prob=0.01)

        ###### Compile if requested ######
        if compile_cfg is not None:
            compile_mdl = compile_cfg['enable']
            if compile_mdl and th_compile is not None:
                self.forward = th_compile(self.forward, **compile_cfg['args'])
            elif compile_mdl:
                print('Could not compile YOLOXHead because torch.compile is not available')
        ##################################

    def initialize_biases(self, prior_prob):
        for conv in self.cls_preds:
            b = conv.bias.view(1, -1)
            b.data.fill_(-math.log((1 - prior_prob) / prior_prob))
            conv.bias = torch.nn.Parameter(b.view(-1), requires_grad=True)

        for conv in self.obj_preds:
            b = conv.bias.view(1, -1)
            b.data.fill_(-math.log((1 - prior_prob) / prior_prob))
            conv.bias = torch.nn.Parameter(b.view(-1), requires_grad=True)

    def forward(self, xin, labels=None):
        train_outputs = []
        inference_outputs = []
        origin_preds = []
        x_shifts = []
        y_shifts = []
        expanded_strides = []

        for k, (cls_conv, reg_conv, stride_this_level, x) in enumerate(
            zip(self.cls_convs, self.reg_convs, self.strides, xin)
        ):
            x = self.stems[k](x)
            cls_x = x
            reg_x = x

            cls_feat = cls_conv(cls_x)
            cls_output = self.cls_preds[k](cls_feat)

            reg_feat = reg_conv(reg_x)
            reg_output = self.reg_preds[k](reg_feat)
            obj_output = self.obj_preds[k](reg_feat)

            if self.training:
                output = torch.cat([reg_output, obj_output, cls_output], 1)
                output, grid = self.get_output_and_grid(
                    output, k, stride_this_level, xin[0].type()
                )
                x_shifts.append(grid[:, :, 0])
                y_shifts.append(grid[:, :, 1])
                expanded_strides.append(
                    torch.zeros(1, grid.shape[1])
                    .fill_(stride_this_level)
                    .type_as(xin[0])
                )
                if self.use_l1:
                    batch_size = reg_output.shape[0]
                    hsize, wsize = reg_output.shape[-2:]
                    reg_output = reg_output.view(
                        batch_size, 1, 4, hsize, wsize
                    )
                    reg_output = reg_output.permute(0, 1, 3, 4, 2).reshape(
                        batch_size, -1, 4
                    )
                    origin_preds.append(reg_output.clone())
                train_outputs.append(output)
            inference_output = torch.cat(
                [reg_output, obj_output.sigmoid(), cls_output.sigmoid()], 1
            )
            inference_outputs.append(inference_output)

        # --------------------------------------------------------
        # Modification: return decoded output also during training
        # --------------------------------------------------------
        # losses = None
        # losses = torch.empty(0)  # for build , losses
      
        self.hw = [x.shape[-2:] for x in inference_outputs]
        # [batch, n_anchors_all, 85]
        outputs = torch.cat(
            [x.flatten(start_dim=2) for x in inference_outputs], dim=2
        ).permute(0, 2, 1)
        if self.decode_in_inference:
            # return self.decode_outputs(outputs), losses
            return self.decode_outputs(outputs)
      

    def get_output_and_grid(self, output, k, stride, dtype):
        grid = self.grids[k]

        batch_size = output.shape[0]
        n_ch = 5 + self.num_classes
        hsize, wsize = output.shape[-2:]
        if grid.shape[2:4] != output.shape[2:4]:
            yv, xv = torch.meshgrid([torch.arange(hsize), torch.arange(wsize)])
            grid = torch.stack((xv, yv), 2).view(1, 1, hsize, wsize, 2).type(dtype)
            self.grids[k] = grid

        output = output.view(batch_size, 1, n_ch, hsize, wsize)
        output = output.permute(0, 1, 3, 4, 2).reshape(
            batch_size, hsize * wsize, -1
        )
        grid = grid.view(1, -1, 2)
        output[..., :2] = (output[..., :2] + grid) * stride
        output[..., 2:4] = torch.exp(output[..., 2:4]) * stride
        return output, grid

    def decode_outputs(self, outputs):
        if self.output_grids is None:
            assert self.output_strides is None
            dtype = outputs.dtype
            device = outputs.device
            grids = []
            strides = []
            for (hsize, wsize), stride in zip(self.hw, self.strides):
                yv, xv = torch.meshgrid([torch.arange(hsize, device=device, dtype=dtype),
                                         torch.arange(wsize, device=device, dtype=dtype)])
                grid = torch.stack((xv, yv), 2).view(1, -1, 2)
                grids.append(grid)
                shape = grid.shape[:2]
                # strides.append(torch.full((*shape, torch.tensor(1)), stride, device=device, dtype=dtype))
                strides.append(stride * torch.ones((*shape, torch.tensor(1)), device=device, dtype=dtype))
            self.output_grids = torch.cat(grids, dim=1)
            self.output_strides = torch.cat(strides, dim=1)
        outputs = torch.cat([
            (outputs[..., 0:2] + self.output_grids) * self.output_strides,
            torch.exp(outputs[..., 2:4]) * self.output_strides,
            outputs[..., 4:]
        ], dim=-1)
        return outputs

    
    def get_l1_target(self, l1_target, gt, stride, x_shifts, y_shifts, eps=1e-8):
        l1_target[:, 0] = gt[:, 0] / stride - x_shifts
        l1_target[:, 1] = gt[:, 1] / stride - y_shifts
        l1_target[:, 2] = torch.log(gt[:, 2] / stride + eps)
        l1_target[:, 3] = torch.log(gt[:, 3] / stride + eps)
        return l1_target

    
    

    


In [20]:
def build_yolox_head_apu(head_cfg: DictConfig, in_channels: Tuple[int, ...], strides: Tuple[int, ...]):
    head_cfg_dict = OmegaConf.to_container(head_cfg, resolve=True, throw_on_missing=True)
    head_cfg_dict.pop('name')
    head_cfg_dict.pop('version', None)
    head_cfg_dict.update({"in_channels": in_channels})
    head_cfg_dict.update({"strides": strides})
    compile_cfg = head_cfg_dict.pop('compile', None)
    head_cfg_dict.update({"compile_cfg": compile_cfg})
    return YOLOXHead_apu(**head_cfg_dict)

In [21]:
def build_yolox_fpn_apu(fpn_cfg: DictConfig, in_channels: Tuple[int, ...]):
    fpn_cfg_dict = OmegaConf.to_container(fpn_cfg, resolve=True, throw_on_missing=True)
    fpn_name = fpn_cfg_dict.pop('name')
    fpn_cfg_dict.update({"in_channels": in_channels})
    if fpn_name in {'PAFPN', 'pafpn'}:
        compile_cfg = fpn_cfg_dict.pop('compile', None)
        fpn_cfg_dict.update({"compile_cfg": compile_cfg})
        return YOLOPAFPN_apu(**fpn_cfg_dict)
    raise NotImplementedError

In [22]:
class YoloV5Detector_S(pt.nn.Module):
    def __init__(self,
                 model_cfg: DictConfig):
        super().__init__()
        backbone_cfg = model_cfg.backbone
        fpn_cfg = model_cfg.fpn
        head_cfg = model_cfg.head

        self.backbone = YoloDetS()
        in_channels = (128, 256, 512)

        self.fpn = build_yolox_fpn_apu(fpn_cfg, in_channels=in_channels)

        strides = (8, 16, 32)
        self.yolox_head = build_yolox_head_apu(head_cfg, in_channels=in_channels, strides=strides)

    def forward_backbone(self,
                         x: pt.Tensor) :
        
        backbone_features = self.backbone(x)
        return backbone_features

    def forward_detect(self,
                       backbone_features):
        device = pt.device('cpu')
       
        fpn_features = self.fpn(backbone_features)

       
        outputs = self.yolox_head(fpn_features)
       
        return outputs

    def forward(self,
                x: pt.Tensor) :
        backbone_features = self.forward_backbone(x)

        outputs = self.forward_detect(backbone_features=backbone_features)
        return outputs

In [23]:

from collections import OrderedDict
config = OmegaConf.load('../../applications/dvsdetection/st-yolo/config/model/yolodet_apu.yaml')
pt_model = YoloV5Detector_S(model_cfg=config)
checkpoint_file = os.path.join("../../weight_files/dvsdetection/styolo/" + "epoch=001-step=394428-val_AP=0.44.ckpt")
out_path = "../../model_files/dvsdetection/styolo/" + "st_" + "yolov5s"
params = torch.load(checkpoint_file, map_location=torch.device('cpu'))
state_dict = OrderedDict((k[4:], v) for k, v in params['state_dict'].items())
pt_model.load_state_dict(state_dict)



from lynadapter.lyn_compile import model_compile
model_compile(pt_model, out_path,[((1, 20, 256, 320),)])



/tmp/ipykernel_49769/2340192989.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  params = torch.load(checkpoint_file, map_location=torch.device('cpu'))
pytorch_graph.py[l

lyngor build outpath: ../../model_files/dvsdetection/styolo/st_yolov5s


In [24]:
from hydra import initialize, compose
from hydra.core.global_hydra import GlobalHydra
from functools import partial
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import numpy as np
import time
import bbox_visualizer as bbv
import sys
sys.path.append("../../applications/dvsdetection/st-yolo/")
from models.detection.yolox.utils import postprocess
from config.modifier import dynamically_modify_train_config
from data.genx_utils.dataset_streaming import build_streaming_dataset
from data.utils.types import DataType, DatasetMode, DatasetSamplingMode, ObjDetOutput
from modules.data.genx_apu import get_dataloader_kwargs
from einops import rearrange, reduce
from enum import Enum, auto
import matplotlib.pyplot as plt
import cv2
class Mode(Enum):
    TRAIN = auto()
    VAL = auto()
    TEST = auto()

    
# 清理 Hydra 全局状态
GlobalHydra.instance().clear()
initialize(config_path="../../applications/dvsdetection/st-yolo/config", job_name="main")
config_name = 'apu_show_styolov5s'
config = compose(config_name=config_name)
dynamically_modify_train_config(config)
OmegaConf.to_container(config, resolve=True, throw_on_missing=True)

print('------ Configuration ------')
#print(OmegaConf.to_yaml(config))
print('---------------------------')
batch_size_eval = config.batch_size.eval
num_workers_generic = config.hardware.get('num_workers', None)
num_workers_eval = config.hardware.num_workers.get('eval', num_workers_generic)
dataset_config = config.dataset

class PropheseeEvaluator:
    LABELS = 'lables'
    PREDICTIONS = 'predictions'

    def __init__(self, dataset: str, downsample_by_2: bool):
        super().__init__()
        assert dataset in {'gen1', 'gen4'}
        self.dataset = dataset
        self.downsample_by_2 = downsample_by_2

        self._buffer = None
        self._buffer_empty = True
        self._reset_buffer()

    def _reset_buffer(self):
        self._buffer_empty = True
        self._buffer = {
            self.LABELS: list(),
            self.PREDICTIONS: list(),
        }

    def _add_to_buffer(self, key: str, value: List[np.ndarray]):
        assert isinstance(value, list)
        for entry in value:
            assert isinstance(entry, np.ndarray)
        self._buffer_empty = False
        assert self._buffer is not None
        self._buffer[key].extend(value)

    def _get_from_buffer(self, key: str) -> List[np.ndarray]:
        assert not self._buffer_empty
        assert self._buffer is not None
        return self._buffer[key]

    def add_predictions(self, predictions: List[np.ndarray]):
        self._add_to_buffer(self.PREDICTIONS, predictions)

    def add_labels(self, labels: List[np.ndarray]):
        self._add_to_buffer(self.LABELS, labels)

    def reset_buffer(self) -> None:
        # E.g. call in on_validation_epoch_start
        self._reset_buffer()

    def has_data(self):
        return not self._buffer_empty

    def evaluate_buffer(self, img_height: int, img_width: int) :
        # e.g call in on_validation_epoch_end
        if self._buffer_empty:
            warn("Attempt to use prophesee evaluation buffer, but it is empty", UserWarning, stacklevel=2)
            return

        labels = self._get_from_buffer(self.LABELS)
        predictions = self._get_from_buffer(self.PREDICTIONS)
        assert len(labels) == len(predictions)
        metrics = evaluate_list(result_boxes_list=predictions,
                                gt_boxes_list=labels,
                                height=img_height,
                                width=img_width,
                                apply_bbox_filters=True,
                                downsampled_by_2=self.downsample_by_2,
                                camera=self.dataset)
        return metrics

class InputPadderFromShape:
    def __init__(self, desired_hw: Tuple[int, int], mode: str = 'constant', value: int = 0, type: str = 'corner'):
        """
        :param desired_hw: Desired height and width
        :param mode: See torch.nn.functional.pad
        :param value:  See torch.nn.functional.pad
        :param type: "corner": add zero to bottom and right
        """
        assert isinstance(desired_hw, tuple)
        assert len(desired_hw) == 2
        assert desired_hw[0] % 4 == 0, 'Required for token mask padding'
        assert desired_hw[1] % 4 == 0, 'Required for token mask padding'
        assert type in {'corner'}

        self.desired_hw = desired_hw
        self.mode = mode
        self.value = value
        self.type = type
        self._pad_ev_repr = None
        self._pad_token_mask = None

    @staticmethod
    def _pad_tensor_impl(input_tensor, desired_hw, mode, value):
        assert isinstance(input_tensor, pt.Tensor)

        ht, wd = input_tensor.shape[-2:]
        ht_des, wd_des = desired_hw
        assert ht <= ht_des
        assert wd <= wd_des

        pad_left = 0
        pad_right = wd_des - wd
        pad_top = 0
        pad_bottom = ht_des - ht

        pad = [pad_left, pad_right, pad_top, pad_bottom]
        return F.pad(input_tensor, pad=pad, mode=mode, value=value if mode == 'constant' else None), pad

    def pad_tensor_ev_repr(self, ev_repr: pt.Tensor) -> pt.Tensor:
        padded_ev_repr, pad = self._pad_tensor_impl(input_tensor=ev_repr, desired_hw=self.desired_hw,
                                                    mode=self.mode, value=self.value)
        if self._pad_ev_repr is None:
            self._pad_ev_repr = pad
        else:
            assert self._pad_ev_repr == pad
        return padded_ev_repr

    def pad_token_mask(self, token_mask: pt.Tensor):
        assert isinstance(token_mask, pt.Tensor)

        desired_hw = tuple(x // 4 for x in self.desired_hw)
        padded_token_mask, pad = self._pad_tensor_impl(input_tensor=token_mask, desired_hw=desired_hw,
                                                       mode='constant', value=0)
        if self._pad_token_mask is None:
            self._pad_token_mask = pad
        else:
            assert self._pad_token_mask == pad
        return padded_token_mask

BBOX_DTYPE = np.dtype({'names': ['t', 'x', 'y', 'w', 'h', 'class_id', 'track_id', 'class_confidence'],
                       'formats': ['<i8', '<f4', '<f4', '<f4', '<f4', '<u4', '<u4', '<f4'],
                       'offsets': [0, 8, 12, 16, 20, 24, 28, 32], 'itemsize': 40})
def to_prophesee_show(yolox_pred_list) -> \
        Tuple[List[np.ndarray], List[np.ndarray]]:
    yolox_pred_list_proph = []
    for yolox_preds in yolox_pred_list:
        # TODO: use loaded_label_to_prophesee func here
        # --- YOLOX PREDICTIONS ---
        # Assumes batch of post-processed predictions from YoloX Head.
        # See postprocessing: https://github.com/Megvii-BaseDetection/YOLOX/blob/a5bb5ab12a61b8a25a5c3c11ae6f06397eb9b296/yolox/utils/boxes.py#L32
        # Detections ordered as (x1, y1, x2, y2, obj_conf, class_conf, class_pred)
        num_pred = 0 if yolox_preds is None else yolox_preds.shape[0]
        yolox_pred_proph = np.zeros((num_pred,), dtype=BBOX_DTYPE)
        if num_pred > 0:
            yolox_preds = yolox_preds.detach().cpu().numpy()
            assert yolox_preds.shape == (num_pred, 7)
            yolox_pred_proph['t'] = np.ones((num_pred,), dtype=BBOX_DTYPE['t'])
            yolox_pred_proph['x'] = np.asarray(yolox_preds[:, 0], dtype=BBOX_DTYPE['x'])
            yolox_pred_proph['y'] = np.asarray(yolox_preds[:, 1], dtype=BBOX_DTYPE['y'])
            yolox_pred_proph['w'] = np.asarray(yolox_preds[:, 2] - yolox_preds[:, 0], dtype=BBOX_DTYPE['w'])
            yolox_pred_proph['h'] = np.asarray(yolox_preds[:, 3] - yolox_preds[:, 1], dtype=BBOX_DTYPE['h'])
            yolox_pred_proph['class_id'] = np.asarray(yolox_preds[:, 6], dtype=BBOX_DTYPE['class_id'])
            yolox_pred_proph['class_confidence'] = np.asarray(yolox_preds[:, 5], dtype=BBOX_DTYPE['class_confidence'])
        yolox_pred_list_proph.append(yolox_pred_proph)

    return yolox_pred_list_proph


def ev_repr_to_img(x: np.ndarray):
    ch, ht, wd = x.shape[-3:]
    assert ch > 1 and ch % 2 == 0
    ev_repr_reshaped = rearrange(x, '(posneg C) H W -> posneg C H W', posneg=2)
    img_neg = np.asarray(reduce(ev_repr_reshaped[0], 'C H W -> H W', 'sum'), dtype='int32')
    img_pos = np.asarray(reduce(ev_repr_reshaped[1], 'C H W -> H W', 'sum'), dtype='int32')
    img_diff = img_pos - img_neg
    img = 127 * np.ones((ht, wd, 3), dtype=np.uint8)
    img[img_diff > 0] = 255
    img[img_diff < 0] = 0
    return img

def filter_predictions(yolox_preds_proph, img):
    image_shape = img.shape

    # Define the color values
    background_color = 127
    black_color = 0
    white_color = 255

    # Define the threshold values for black and white color counts and class confidence
    color_threshold = 20
    confidence_threshold = 0.8

    yolox_preds_proph_copy = np.copy(yolox_preds_proph)[0].tolist()

    # Iterate over each prediction in yolox_preds_proph_copy in reverse order
    for i in reversed(range(len(yolox_preds_proph_copy))):
        # Get the x, y, w, h values for the bounding box
        x, y, w, h = yolox_preds_proph_copy[i][1:5]

        # Calculate the coordinates of the top-left and bottom-right corner of the bounding box
        x_min = int(x)
        y_min = int(y)
        x_max = int(x + w)
        y_max = int(y + h)

        # Check if the bounding box is within the image boundaries
        if x_min >= 0 and y_min >= 0 and x_max < image_shape[1] and y_max < image_shape[0]:
            # Extract the region of interest from the image
            roi = img[y_min:y_max, x_min:x_max, 0]

            # Count the number of black and white pixels in the region of interest
            color_count = np.sum(roi == black_color) + np.sum(roi == white_color)

            # Check if color count is below the threshold and class confidence is below the threshold
            if color_count < color_threshold and yolox_preds_proph_copy[i][-1] < confidence_threshold:
                # Remove the prediction by setting all values to 0
                yolox_preds_proph_copy.pop(i)

    # Update yolox_preds_proph with the filtered predictions
    yolox_preds_proph[0] = np.array(yolox_preds_proph_copy, dtype=yolox_preds_proph[0].dtype)

    return yolox_preds_proph
LABELMAP_GEN1 = ("car", "pedestrian")
def draw_bboxes_bbv(img, boxes, save_id, labelmap=LABELMAP_GEN1, is_pred=False) -> np.ndarray:
    """
    draw bboxes in the image img
    """
    colors = cv2.applyColorMap(np.arange(0, 255).astype(np.uint8), cv2.COLORMAP_HSV)
    colors = [tuple(*item) for item in colors.tolist()]

   
    classid2colors = {
        0: (255, 255, 0),  # car -> yellow (rgb)
        1: (0, 0, 255),  # ped -> blue (rgb)
    }
    scale_multiplier = 4


    add_score = True
    ht, wd, ch = img.shape
    dim_new_wh = (int(wd * scale_multiplier), int(ht * scale_multiplier))
    if scale_multiplier != 1:
        img = cv2.resize(img, dim_new_wh, interpolation=cv2.INTER_AREA)
    if boxes is not None:
        for i in range(boxes.shape[0]):
            pt1 = (int(boxes['x'][i]), int(boxes['y'][i]))
            size = (int(boxes['w'][i]), int(boxes['h'][i]))
            pt2 = (pt1[0] + size[0], pt1[1] + size[1])
            bbox = (pt1[0], pt1[1], pt2[0], pt2[1])
            bbox = tuple(x * scale_multiplier for x in bbox)

            score = boxes['class_confidence'][i]
            class_id = boxes['class_id'][i]
            class_name = LABELMAP_GEN1[class_id % len(LABELMAP_GEN1)]
            bbox_txt = class_name
            if add_score:
                bbox_txt += f' {score:.2f}'
            color_tuple_rgb = classid2colors[class_id]
            img = bbv.draw_rectangle(img, bbox, bbox_color=color_tuple_rgb)
            img = bbv.add_label(img, bbox_txt, bbox, text_bg_color=color_tuple_rgb, top=True)

    visualize_dir = './visualize'

    if not os.path.exists(visualize_dir):
        os.makedirs(visualize_dir)
        pred_dir = os.path.join(visualize_dir, 'pred')
        os.makedirs(pred_dir)

    save_id = '{:04d}'.format(save_id)
    if is_pred:
        save_path = './visualize/pred/' + str(save_id) + ".png"
    else:
        save_path =  './visualize/label/' + str(save_id) + ".png"
    cv2.imwrite(save_path, img)
    return img

eval_sampling_mode = 'stream'

build_eval_dataset = partial(build_streaming_dataset,
                             batch_size=batch_size_eval,
                             num_workers=num_workers_eval)

test_dataset = build_eval_dataset(dataset_mode=DatasetMode.TESTING, dataset_config=dataset_config)

test_dataloader = DataLoader(**get_dataloader_kwargs(
    dataset=test_dataset, sampling_mode=eval_sampling_mode, dataset_mode=DatasetMode.TESTING,
    dataset_config=dataset_config, batch_size=batch_size_eval,
    num_workers=num_workers_eval))

### initialize
dataset_name = config.dataset.name
mode = Mode.TEST
mode_2_hw: Dict[Mode, Optional[Tuple[int, int]]] = {}
mode_2_batch_size: Dict[Mode, Optional[int]] = {}
mode_2_psee_evaluator: Dict[Mode, Optional[PropheseeEvaluator]] = {}
mode_2_sampling_mode: Dict[Mode, DatasetSamplingMode] = {}
dataset_eval_sampling = config.dataset.eval.sampling
assert dataset_eval_sampling in (DatasetSamplingMode.STREAM, DatasetSamplingMode.RANDOM)
mode_2_psee_evaluator[mode] = PropheseeEvaluator(
    dataset=dataset_name, downsample_by_2=config.dataset.downsample_by_factor_2)
mode_2_sampling_mode[Mode.TEST] = dataset_eval_sampling
mode_2_hw[mode] = None
mode_2_batch_size[mode] = None
### initialize finish

from lynadapter.lyn_sdk_model import ApuRun_Single
device =0
path_model = os.path.join(out_path, "Net_0")
arun = ApuRun_Single(device, path_model)

num_iterations = len(test_dataset)
n = 0
show_id = 1
t_total = 0

for i,  val_batch in enumerate(test_dataloader):
    if i==show_id:
        n += 1
        data = val_batch['data']
        worker_id = val_batch['worker_id']
        ev_tensor_sequence = data[DataType.EV_REPR]
        sparse_obj_labels = data[DataType.OBJLABELS_SEQ]
        is_first_sample = data[DataType.IS_FIRST_SAMPLE]
    
        sequence_len = len(ev_tensor_sequence)
        assert sequence_len > 0
        batch_size = len(sparse_obj_labels[0])
        if mode_2_batch_size[mode] is None:
            mode_2_batch_size[mode] = batch_size
        else:
            assert mode_2_batch_size[mode] == batch_size
    
        in_res_hw = tuple(config.model.backbone.in_res_hw)
        input_padder = InputPadderFromShape(desired_hw=in_res_hw)
        print(sequence_len)
        plt.figure(figsize=(18, 36))
        for tidx in range(sequence_len):
        
        
            collect_predictions = (tidx == sequence_len - 1) or \
                                  (mode_2_sampling_mode[mode] == DatasetSamplingMode.STREAM)
            ev_tensors = ev_tensor_sequence[tidx]
            ev_tensors = ev_tensors.to(dtype=torch.float32)
            ev_tensors = input_padder.pad_tensor_ev_repr(ev_tensors)
            if mode_2_hw[mode] is None:
                mode_2_hw[mode] = tuple(ev_tensors.shape[-2:])
            else:
                assert mode_2_hw[mode] == ev_tensors.shape[-2:]
    
            data_img = np.array(ev_tensors).astype(np.float32)
            t1 = time.time()
            if is_first_sample:
                output = arun.run(data_img, 0)  # reset
            else:
                output = arun.run(data_img, 1)
            t2 = time.time()
            t_total += (t2 - t1)
            predictions = torch.from_numpy(output[0])
    
            pred_processed = postprocess(prediction=predictions,
                                         num_classes=config.model.head.num_classes,
                                         conf_thre=config.model.postprocess.confidence_threshold,
                                         nms_thre=config.model.postprocess.nms_threshold)
    
            yolox_preds_proph = to_prophesee_show(pred_processed)
    
            ev_img = ev_repr_to_img(ev_tensors[0].cpu().numpy())
            prediction_img = ev_img.copy()
            yolox_preds_proph = filter_predictions(yolox_preds_proph, prediction_img)           
            boxes = yolox_preds_proph[0]
            colors = cv2.applyColorMap(np.arange(0, 255).astype(np.uint8), cv2.COLORMAP_HSV)
            colors = [tuple(*item) for item in colors.tolist()]
            classid2colors = {
                0: (255, 255, 0),  # car -> yellow (rgb)
                1: (0, 0, 255),  # ped -> blue (rgb)
            }
            scale_multiplier = 4      
        
            add_score = True
            ht, wd, ch = prediction_img.shape
            dim_new_wh = (int(wd * scale_multiplier), int(ht * scale_multiplier))
            if scale_multiplier != 1:
                prediction_img = cv2.resize(prediction_img, dim_new_wh, interpolation=cv2.INTER_AREA)
            if boxes is not None:
                for i in range(boxes.shape[0]):
                    pt1 = (int(boxes['x'][i]), int(boxes['y'][i]))
                    size = (int(boxes['w'][i]), int(boxes['h'][i]))
                    pt2 = (pt1[0] + size[0], pt1[1] + size[1])
                    bbox = (pt1[0], pt1[1], pt2[0], pt2[1])
                    bbox = tuple(x * scale_multiplier for x in bbox)
        
                    score = boxes['class_confidence'][i]
                    class_id = boxes['class_id'][i]
                    class_name = LABELMAP_GEN1[class_id % len(LABELMAP_GEN1)]
                    bbox_txt = class_name
                    if add_score:
                        bbox_txt += f' {score:.2f}'
                    color_tuple_rgb = classid2colors[class_id]
                    prediction_img = bbv.draw_rectangle(prediction_img, bbox, bbox_color=color_tuple_rgb)
                    prediction_img = bbv.add_label(prediction_img, bbox_txt, bbox, text_bg_color=color_tuple_rgb, top=True)
        
            plt.subplot(6, 2, tidx+1)
           
            plt.imshow(prediction_img)
        plt.show()
        break
arun.apu_unload()        
print('traverse finished')

ModuleNotFoundError: No module named 'models'